In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
RUTA_DATOS = "/content/ENSANUT_ENTRENAMIENTO.csv"
df = pd.read_csv(RUTA_DATOS)
print("="*60)
print("DATASET CARGADO")
print("="*60)
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("\n")

DATASET CARGADO
Filas: 20510
Columnas: 115




In [ ]:
import pandas as pd
!pip install pandas_ods_reader
from pandas_ods_reader import read_ods
DICCIONARIO_FINAL = "/content/dicentre.ods"
dic_final = read_ods(
    DICCIONARIO_FINAL,
    1
)
diccionario_final = dict(
zip(
dic_final["Nombre del campo"],
dic_final["Descripción del campo"]
)
)

def renombrar_columnas(df, diccionario):
    nuevas_columnas = []
    for columna in df.columns:
        if columna in diccionario:
            nuevas_columnas.append(diccionario[columna])
        else:
            nuevas_columnas.append(columna)
    df.columns = nuevas_columnas
    return df


final_limpio = renombrar_columnas(
    df,
    diccionario_final
)

display(final_limpio.head())

,area,provincia,orden_hijo,sexo,embarazo_planeado,embarazo_deseado_pareja,control_prenatal,lugar_control_prenatal,consumo_micronutrientes_embarazo,frecuencia_micronutrientes,...,lugar_vacuna_srp_1,lugar_vacuna_srp_2,region_madre,etnia_madre,edad_meses,grupo_edad_meses,nivel_instruccion_madre,desnutricion_cronica,factor_expansion,estrato
0,Urbano,1.0,1.0,Mujer,Quería Esperar Mas Tiempo ?,Quería Esperar Mas Tiempo ?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,No Sabe/ No Responde,No Sabe/ No Responde,Sierra,Mestizo,18.0,12-18,Superior,0.0,39.997669,2713.0
1,Urbano,1.0,1.0,Mujer,Quería Esperar Mas Tiempo ?,Tener Ese Hijo?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Aún No La Recibe,Aún No La Recibe,Sierra,Mestizo,8.0,0-11,Educación Básica,0.0,45.630718,2713.0
2,Urbano,1.0,1.0,Mujer,No Quería Más Hijos?,No Quería Más Hijos?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Aún No La Recibe,Aún No La Recibe,Sierra,Mestizo,20.0,19-23,Educación Básica,1.0,41.165138,2713.0
3,Urbano,1.0,1.0,Mujer,Tener Ese Hijo?,Tener Ese Hijo?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Aún No La Recibe,Aún No La Recibe,Sierra,Mestizo,1.0,0-11,Educación Media/Bachillerato,0.0,39.997669,2713.0
4,Urbano,1.0,1.0,Hombre,Quería Esperar Mas Tiempo ?,Tener Ese Hijo?,Si,Clínica/Consultorio Privado,Ácido Fólico,Diaria,...,Establecimientos De Salud Del Msp,Establecimientos De Salud Del Msp,Sierra,Mestizo,49.0,48-59,Superior,1.0,147.648470,2713.0


In [ ]:
# Guardar dataset con nombres compatibles con el modelo

final_limpio.to_csv(
    "ENSANUT_MODELO.csv",
    index=False,
    encoding="utf-8"
)

print("Dataset guardado correctamente")
print(final_limpio.shape)

Dataset guardado correctamente
(20510, 115)


In [ ]:
TARGET = "desnutricion_cronica"
print("Variable objetivo:")
print(TARGET)
# ==========================================
# SEPARAR VARIABLES
# ==========================================

X = df.drop(columns=[TARGET])
y = df[TARGET]

Variable objetivo:
desnutricion_cronica


In [ ]:
# ==========================================
# VARIABLES NUMÉRICAS
# ==========================================

variables_numericas = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# ==========================================
# VARIABLES CATEGÓRICAS
# ==========================================

variables_categoricas = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

/tmp/ipykernel_5917/2925543495.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  variables_categoricas = X.select_dtypes(


In [ ]:
# ==========================================
# INFORMACIÓN
# ==========================================
print("="*60)
print("RESUMEN")
print("="*60)
print(f"Variables predictoras : {X.shape[1]}")
print(f"Variables numéricas   : {len(variables_numericas)}")
print(f"Variables categóricas : {len(variables_categoricas)}")
print("\n")
print("Primeras variables numéricas:")
print(variables_numericas[:15])
print("\n")
print("Primeras variables categóricas:")
print(variables_categoricas[:15])

RESUMEN
Variables predictoras : 114
Variables numéricas   : 21
Variables categóricas : 93


Primeras variables numéricas:
['provincia', 'orden_hijo', 'numero_examenes_vih', 'dosis_tetanos', 'semanas_primer_control', 'numero_controles_prenatales', 'talla_nacer_cm', 'perimetro_cefalico_cm', 'peso_nacer_gramos', 'tiempo_primer_control_posparto', 'tiempo_primer_control_nino', 'controles_0_1_anio', 'controles_1_2_anios', 'controles_2_5_anios', 'numero_controles_carnet']


Primeras variables categóricas:
['area', 'sexo', 'embarazo_planeado', 'embarazo_deseado_pareja', 'control_prenatal', 'lugar_control_prenatal', 'consumo_micronutrientes_embarazo', 'frecuencia_micronutrientes', 'control_peso_embarazo', 'medicion_altura_uterina', 'examen_vih_embarazo', 'control_presion_embarazo', 'examen_sangre_embarazo', 'examen_orina_embarazo', 'examen_sifilis_embarazo']


In [ ]:
# ==========================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ==========================================

print("="*60)
print("DESNUTRICIÓN CRÓNICA")
print("="*60)
print(y.value_counts())
print("\n")
print("Porcentajes")
print(
    round(
        y.value_counts(normalize=True)*100,
        2
    )
)

DESNUTRICIÓN CRÓNICA
desnutricion_cronica
0.0    15950
1.0     4560
Name: count, dtype: int64


Porcentajes
desnutricion_cronica
0.0    77.77
1.0    22.23
Name: proportion, dtype: float64


In [ ]:
# ==============================================
# PREPROCESAMIENTO
# ==============================================

transformador_numerico = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

transformador_categorico = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocesador = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, variables_numericas),
        ("cat", transformador_categorico, variables_categoricas)
    ]
)

In [ ]:
# ==============================================
# DIVISIÓN DEL DATASET
# ==============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print()
print("Entrenamiento :",X_train.shape)
print("Prueba :",X_test.shape)


Entrenamiento : (16408, 114)
Prueba : (4102, 114)


In [ ]:
df_train = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
df_test = pd.concat([X_test, y_test.reset_index(drop=True)], axis=1)
df_train.to_csv('train.csv', index=False)
df_test.to_csv('test.csv', index=False)

In [ ]:
display(df_train.head())
display(df_test.head())

,area,provincia,orden_hijo,sexo,embarazo_planeado,embarazo_deseado_pareja,control_prenatal,lugar_control_prenatal,consumo_micronutrientes_embarazo,frecuencia_micronutrientes,...,lugar_vacuna_srp_1,lugar_vacuna_srp_2,region_madre,etnia_madre,edad_meses,grupo_edad_meses,nivel_instruccion_madre,factor_expansion,estrato,desnutricion_cronica
2552,Rural,5.0,1.0,Hombre,No Quería Más Hijos?,Quería Esperar Mas Tiempo ?,Si,Establecimientos De Salud Del Msp,Ninguno,Diaria,...,Aún No La Recibe,Aún No La Recibe,Sierra,Mestizo,15.0,12-18,Educación Básica,52.463398,521.0,0.0
13182,Rural,15.0,1.0,Mujer,Tener Ese Hijo?,Tener Ese Hijo?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Pasando Un Día,...,Establecimientos De Salud Del Msp,Aún No La Recibe,Amazonía,Indígena,17.0,12-18,Superior,5.704193,1522.0,0.0
1286,Urbano,3.0,1.0,Mujer,Tener Ese Hijo?,Tener Ese Hijo?,Si,Clínica/Consultorio Privado,Hierro Más Ácido Fólico?,Diaria,...,Aún No La Recibe,Aún No La Recibe,Sierra,Mestizo,16.0,12-18,Superior,14.552987,313.0,0.0
7885,Urbano,10.0,1.0,Hombre,Tener Ese Hijo?,Tener Ese Hijo?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Aún No La Recibe,Aún No La Recibe,Sierra,Mestizo,9.0,0-11,Educación Media/Bachillerato,73.878181,1012.0,0.0
6614,Urbano,9.0,2.0,Hombre,Tener Ese Hijo?,Tener Ese Hijo?,Si,Hospital/Clínica/Dispensario Del Iess,Hierro Más Ácido Fólico?,Diaria,...,Establecimientos De Salud Del Msp,Establecimientos De Salud Del Msp,Costa,Mestizo,22.0,19-23,Superior,83.509911,2611.0,0.0


,area,provincia,orden_hijo,sexo,embarazo_planeado,embarazo_deseado_pareja,control_prenatal,lugar_control_prenatal,consumo_micronutrientes_embarazo,frecuencia_micronutrientes,...,lugar_vacuna_srp_1,lugar_vacuna_srp_2,region_madre,etnia_madre,edad_meses,grupo_edad_meses,nivel_instruccion_madre,factor_expansion,estrato,desnutricion_cronica
20420,Urbano,90.0,1.0,Mujer,Tener Ese Hijo?,Tener Ese Hijo?,Si,Clínica/Consultorio Privado,Hierro Más Ácido Fólico?,Diaria,...,Establecimientos De Salud Del Msp,Aún No La Recibe,Costa,Mestizo,17.0,12-18,Educación Media/Bachillerato,34.566502,9022.0,NaN
16790,Rural,21.0,1.0,Mujer,Tener Ese Hijo?,No Quería Más Hijos?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Establecimientos De Salud Del Msp,Establecimientos De Salud Del Msp,Amazonía,Mestizo,35.0,31-35,Educación Media/Bachillerato,20.875664,2123.0,NaN
14808,Urbano,17.0,1.0,Mujer,Tener Ese Hijo?,Tener Ese Hijo?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Establecimientos De Salud Del Msp,Establecimientos De Salud Del Msp,Sierra,Mestizo,32.0,31-35,Superior,432.359470,1713.0,NaN
849,Rural,2.0,1.0,Mujer,Tener Ese Hijo?,Tener Ese Hijo?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Establecimientos De Salud Del Msp,Establecimientos De Salud Del Msp,Sierra,Mestizo,50.0,48-59,Educación Básica,18.451191,222.0,1.0
3315,Urbano,6.0,1.0,Mujer,Tener Ese Hijo?,Tener Ese Hijo?,Si,Establecimientos De Salud Del Msp,Hierro Más Ácido Fólico?,Diaria,...,Aún No La Recibe,Aún No La Recibe,Sierra,Indígena,4.0,0-11,Educación Media/Bachillerato,44.908504,612.0,0.0


In [ ]:
# ==============================================
# MODELOS
# ==============================================

modelos = {
    "Logistic Regression":
        LogisticRegression(max_iter=500),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "Random Forest":
        RandomForestClassifier(
            random_state=42
        )
}

In [ ]:
# ==============================================
# ENTRENAMIENTO
# ==============================================

resultados = []
for nombre, modelo in modelos.items():
    print("="*60)
    print(nombre)
    print("="*60)
    pipeline = Pipeline([
        ("preprocesador", preprocesador),
        ("modelo", modelo)
    ])
    pipeline.fit(X_train,y_train)
    predicciones = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test,predicciones)
    precision = precision_score(
        y_test,
        predicciones,
        average="weighted"
    )
    recall = recall_score(
        y_test,
        predicciones,
        average="weighted"
    )
    f1 = f1_score(
        y_test,
        predicciones,
        average="weighted"
    )
    resultados.append([
        nombre,
        accuracy,
        precision,
        recall,
        f1
    ])

Logistic Regression


In [ ]:
# ==============================================
# RESULTADOS
# ==============================================

resultados = pd.DataFrame(
    resultados,
    columns=[
        "Modelo",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)
print()
print("="*70)
print(resultados.sort_values(
    by="F1 Score",
    ascending=False
))
print("="*70)

In [ ]:
!pip install xgboost
!pip install lightgbm
!pip install catboost
!pip install joblib

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import joblib

In [ ]:
modelos = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    ),

    "LightGBM": LGBMClassifier(
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        verbose=0,
        random_state=42
    )

}

In [ ]:
# ==============================================
# ENTRENAMIENTO
# ==============================================

resultados = []
pipelines = {}
for nombre, modelo in modelos.items():
    print("="*60)
    print(nombre)
    print("="*60)
    pipeline = Pipeline([
        ("preprocesador", preprocesador),
        ("modelo", modelo)
    ])
    pipeline.fit(X_train,y_train)
    pipelines[nombre] = pipeline
    predicciones = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test,predicciones)
    precision = precision_score(
        y_test,
        predicciones,
        average="weighted"
    )
    recall = recall_score(
        y_test,
        predicciones,
        average="weighted"
    )
    f1 = f1_score(
        y_test,
        predicciones,
        average="weighted"
    )
    resultados.append([
        nombre,
        accuracy,
        precision,
        recall,
        f1
    ])

In [ ]:
# ==============================================
# RESULTADOS
# ==============================================

resultados = pd.DataFrame(
    resultados,
    columns=[
        "Modelo",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)
print()
print("="*70)
print(resultados.sort_values(
    by="F1 Score",
    ascending=False
))
print("="*70)

In [ ]:
mejor_modelo = resultados.sort_values(
    by="F1 Score",
    ascending=False
).iloc[0]

print("\n")
print("="*60)
print("MEJOR MODELO")
print("="*60)
print(mejor_modelo)

In [ ]:
nombre_mejor = mejor_modelo["Modelo"]

pipeline_final = pipelines[nombre_mejor]

joblib.dump(
    pipeline_final,
    "/content/mejor_modelo.joblib"
)

print("\nModelo guardado correctamente.")

# **OPTIMIZACIÓN**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

In [ ]:
pipeline_xgb = pipelines["XGBoost"]

In [ ]:
parametros = {

    "modelo__n_estimators":[
        100,
        200,
        300,
        400,
        500
    ],

    "modelo__max_depth":[
        3,
        4,
        5,
        6,
        7,
        8
    ],

    "modelo__learning_rate":[
        0.01,
        0.03,
        0.05,
        0.1,
        0.2
    ],

    "modelo__subsample":[
        0.6,
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "modelo__colsample_bytree":[
        0.6,
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "modelo__gamma":[
        0,
        0.1,
        0.2,
        0.3,
        0.5
    ],

    "modelo__min_child_weight":[
        1,
        3,
        5,
        7
    ]

}

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
busqueda = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=parametros,
    n_iter=40,
    scoring="f1_weighted",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [ ]:
busqueda.fit(X_train,y_train)

In [ ]:
print(busqueda.best_params_)

In [ ]:
print(busqueda.best_score_)

In [ ]:
pipeline_optimizado = busqueda.best_estimator_
joblib.dump(
    pipeline_optimizado,
    "/content/xgboost_optimizado.joblib"
)

In [ ]:
# ==========================================================
# EVALUACIÓN DEL MODELO OPTIMIZADO
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
# Predicciones
y_pred = pipeline_optimizado.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    average="weighted"
)
recall = recall_score(
    y_test,
    y_pred,
    average="weighted"
)
f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)
print("="*60)
print("RESULTADOS DEL MODELO OPTIMIZADO")
print("="*60)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
# ==========================================================
# MATRIZ DE CONFUSIÓN
# ==========================================================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    y_test,
    y_pred
)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)
fig, ax = plt.subplots(figsize=(6,6))
disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False
)
plt.title("Matriz de Confusión")
plt.show()

In [ ]:
# ==========================================================
# REPORTE DE CLASIFICACIÓN
# ==========================================================

from sklearn.metrics import classification_report
print("="*60)
print("CLASIFICATION REPORT")
print("="*60)

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
print(y.unique())

In [ ]:
# ==========================================================
# CURVA ROC
# ==========================================================

from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
probabilidades = pipeline_optimizado.predict_proba(X_test)[:,1]
auc = roc_auc_score(
    y_test,
    probabilidades
)
fpr, tpr, thresholds = roc_curve(
    y_test,
    probabilidades
)
plt.figure(figsize=(7,7))
plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.3f}"
)
plt.plot(
    [0,1],
    [0,1],
    "--"
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.show()
print(f"AUC = {auc:.4f}")

In [ ]:
# ==========================================================
# IMPORTANCIA DE VARIABLES
# ==========================================================

modelo = pipeline_optimizado.named_steps["modelo"]
preprocesador = pipeline_optimizado.named_steps["preprocesador"]

In [ ]:
import numpy as np
# Variables numéricas
variables_num = variables_numericas
# Variables categóricas codificadas
encoder = preprocesador.named_transformers_[
    "cat"
].named_steps["encoder"]
variables_cat = encoder.get_feature_names_out(
    variables_categoricas
)
# Unimos todas
variables_finales = np.concatenate(
    [
        variables_num,
        variables_cat
    ]
)

In [ ]:
importancias = modelo.feature_importances_

In [ ]:
df_importancia = pd.DataFrame({
    "Variable": variables_finales,
    "Importancia": importancias
})
df_importancia = df_importancia.sort_values(
    by="Importancia",
    ascending=False
)
display(df_importancia.head(20))

In [ ]:
plt.figure(figsize=(10,8))
plt.barh(
    df_importancia["Variable"][:20][::-1],
    df_importancia["Importancia"][:20][::-1]
)
plt.xlabel("Importancia")
plt.ylabel("Variables")
plt.title("Top 20 Variables Más Importantes")
plt.show()

In [ ]:
df_importancia.to_excel(
    "/content/Importancia_variables.xlsx",
    index=False
)

In [ ]:
# ==========================================================
# MODELO FINAL
# ==========================================================

joblib.dump(
    pipeline_optimizado,
    "/content/modelo_final_api.joblib"
)
print("="*60)
print("MODELO FINAL GUARDADO")
print("="*60)